In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("ps08.ipynb")

# PS8 — Metabolism, Retroactivity and Control
### BioE 147/247 · Fall 2026

**Out:** Thursday, November 5 · **Due:** Thursday, November 19, 11:59 pm
**Covers:** Sessions 18, 20, 21 and 22 · **50 points** · **Question 5** is required for **BioE 247** (58 total) and is extra credit for **BioE 147** (up to +8)

---

One thread, and it is a column in a table.

Question 1 asks you to choose an implementation medium by comparing gate
families on orthogonality, speed and **load**. The first two columns you can
fill from session 18. **The third you cannot**, and the rest of this set is
what it takes to fill it.

Session 20 writes a coupling down **exactly** — stoichiometry, with no rate
constants in it — and finds that exactness is not the same as enough.
Session 21 takes a coupling your diagram **mis-draws**, an arrow that is
really a binding site, and prices it. Session 22 builds a controller that
rejects a disturbance **without modeling it** — and then dilution, which is
session 19's problem, degrades the guarantee.

So: choose a medium, then find out what the third column costs you.

**Collaboration is encouraged.** Discuss, argue, work at a whiteboard
together, then write your own solution and your own code. Record who you
worked with below.

**If you used an LLM**, say so briefly and say what for. The conditions are
that you can explain anything you submit and that the code you submit runs.

**A note on the visible tests.** They check *properties* — limits,
invariances, symmetries. A green visible check means "not obviously broken",
not "right".

Fill in the two cells below before you start.

In [ ]:
COLLABORATORS = ""   # e.g. "worked with J. Chen on Q3"
AI_USE = ""          # e.g. "used an LLM to check my algebra in Q4"

## Setup

In [ ]:
# ---------------------------------------------------------------------------
# SETUP — run this cell first, every time.
#
# DataHub / local : finds the repository root and puts it on the import path.
# Google Colab    : clones the repository first, because Colab opens this
#                   notebook on its own, without the posb package beside it.
# ---------------------------------------------------------------------------
import os
import sys

if "google.colab" in sys.modules:
    if not os.path.exists("posb2026"):
        !git clone -q https://github.com/ArkinLaboratory/posb2026.git
    sys.path.insert(0, os.path.abspath("posb2026"))
else:
    _d = os.getcwd()
    while _d != os.path.dirname(_d) and not os.path.isdir(os.path.join(_d, "posb")):
        _d = os.path.dirname(_d)
    sys.path.insert(0, _d)

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

import posb
from posb import Reaction, Model

posb.check_environment()


In [ ]:
from posb.fba import (FluxModel, solve_fba, knockout_growth,
                      flux_ranges, coupled_pairs)
from posb.retroactivity import (retroactivity, steady_state, simulate_load,
                                response_half_time, slowdown, level_shift)
from scipy.integrate import solve_ivp

---
## Question 1 — Choosing a medium  *(6 points)*

You are specifying a circuit and must pick a gate family. This course has
built four: **transcriptional repressors** (promoter occupancy and
autoregulation, sessions 6–7), **recombinases** (sequential logic, session 16),
**CRISPRi** (parts and context, session 17) and **protein / covalent cycles**
(session 18, which also put all four side by side on these three axes).

<!-- BEGIN QUESTION -->

**Q1.** Fill in this table, then argue for one family.

| family | orthogonality | speed | load on the host |
|---|---|---|---|
| transcriptional repressor | | | |
| CRISPRi | | | |
| recombinase | | | |
| protein / covalent | | | |

**(a)** Fill the first two columns from what the course actually measured —
sessions 6–7 for the repressor row, 16–18 for the other three. One phrase per
cell, and name the quantity you are using — not "fast" but *what* is fast and
against what. **One cell in the orthogonality column has no measured value at
all.** Leave it empty and say in a phrase why it is empty.

**(b)** The load column was filled for three of these four families and left
empty for the fourth, because nobody has measured it. Name that family, then
say precisely what you would need to *measure* to fill the cell — not what you
would need to know, what you would need to measure.

**(c)** Choose a family for a circuit that must respond in **under a minute**
and run in a cell that is also expressing a heterologous pathway. Defend it in
three sentences, and name the thing your choice costs you.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

---
## Question 2 — The coupling written down exactly  *(14 points)*

A different network from the one in class. Nine reactions, six balanced
metabolites, and the uptake is capped at **12**:

| reaction | stoichiometry |
|---|---|
| `r_up` | → S6 *(capped at 12)* |
| `r_a` | S6 → 2 T3 |
| `r_b` | S6 → P5 + CO₂ |
| `r_c` | T3 → PEP + ATP |
| `r_d` | PEP → PYR + ATP |
| `r_e` | PYR → 3 CO₂ + 3 ATP |
| `r_f` | PYR → Ace + CO₂ + ATP |
| `r_bio` | 2 T3 + P5 + 4 ATP → |
| `r_atpm` | ATP → |

CO₂ and acetate leave the system and are **not** balanced.

**Q2a.** Build the model. Set `MODEL` to a `FluxModel` over these
reactions, with `CO2_x` and `Ace_x` external, and `BOUNDS` to the uptake cap.

Then set `NULLDIM` to the dimension of the null space of **S** — the number of
degrees of freedom the stoichiometry alone cannot resolve. `FluxModel` will
give you this.

In [ ]:
REACTIONS = [
    ({},                           {"S6": 1},                            "r_up"),
    ({"S6": 1},                    {"T3": 2},                            "r_a"),
    ({"S6": 1},                    {"P5": 1, "CO2_x": 1},                "r_b"),
    ({"T3": 1},                    {"PEP": 1, "ATP": 1},                 "r_c"),
    ({"PEP": 1},                   {"PYR": 1, "ATP": 1},                 "r_d"),
    ({"PYR": 1},                   {"CO2_x": 3, "ATP": 3},               "r_e"),
    ({"PYR": 1},                   {"Ace_x": 1, "CO2_x": 1, "ATP": 1},   "r_f"),
    ({"T3": 2, "P5": 1, "ATP": 4}, {},                                   "r_bio"),
    ({"ATP": 1},                   {},                                   "r_atpm"),
]

...

MODEL.S.shape, NULLDIM

In [ ]:
grader.check("q2a")

**Q2b.** Maximize `r_bio`. Set `BIOMASS` to the optimal value and
`FLUXES` to the flux dictionary that achieves it.

In [ ]:
...

BIOMASS, {k: round(v, 4) for k, v in FLUXES.items()}

In [ ]:
grader.check("q2b")

**Q2c.** Two questions the network answers before any experiment.

**(i)** Set `LETHAL` to the sorted list of reaction names whose single
deletion drives biomass to exactly zero. Consider the seven reactions that are
not `r_up` or `r_bio`.

**(ii)** Set `COUPLED` to the dictionary `coupled_pairs` returns for this
network at these bounds.

In [ ]:
...

LETHAL, COUPLED

In [ ]:
grader.check("q2c")

**Q2d.** Now change the objective. Re-solve the same network
maximising `r_atpm` — ATP production rather than growth — and set `ATP_MAX` to
the optimal value.

In [ ]:
...

ATP_MAX, {k: round(v, 3) for k, v in ATP_FLUXES.items()}

In [ ]:
grader.check("q2d")

<!-- BEGIN QUESTION -->

**Q2e.** Look at what that did.

**(i)** Compare your two flux vectors. In one sentence, what is the network
doing differently when it maximizes ATP, and what has gone to zero?

**(ii)** Biomass maximization is not derived from the network. Say what
assumption about the organism it encodes, and what would have to be true of
the strain for it to be a bad assumption.

**(iii)** Fong & Palsson found eleven predictions wrong out of fifty. Their
paper gives **one** diagnosis for the ones it diagnoses at all: *"Predicted
growth rates were accurate in half the cases for both the ppc⁻ and tpi⁻
strains, suggesting that the reconstructed network may be incomplete."*

Suppose instead you had a strain where the *objective* was wrong — where
selection had not been pushing on growth. Describe a signature in the
predicted-against-observed data that would distinguish that case from a
missing reaction. One or two sentences.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

---
## Question 3 — The coupling the diagram mis-draws  *(16 points)*

A regulator $X$ made at $k$ and removed at $\gamma$, read by a downstream
module with $p_{\text{tot}}$ binding sites and dissociation constant $K_d$:

$$C(X) = \frac{p_{\text{tot}}X}{K_d + X}, \qquad X_T = k/\gamma$$

Throughout this question use $k = 5$, $\gamma = 0.01$ — so $X_T = 500$ — with
$p_{\text{tot}} = 200$ and $K_d = 50$ unless told otherwise.

**Note these are not the numbers from the home guide.** The guide worked the
case where the sites are essentially full. Here they are not, and that is the
question.

<!-- BEGIN QUESTION -->

**Q3a.** Your home guide derived the load term for a simple upstream
module, $f(X) = k - \gamma X$. Now **the upstream module regulates itself**.
Session 7's negatively autoregulated gene:

$$f(X) \;=\; \frac{k}{1 + (X/K)^{2}} \;-\; \gamma X$$

Nothing downstream changes: the same $p_{\text{tot}}$ sites, the same $K_d$.

**(i)** Redo the chain-rule step with this $f$. What is $R(X)$ now? Answer in
one line, and then say — in one sentence, from the structure of the
derivation — why that had to be the answer before you computed anything.

**(ii)** In the **protected** model, does the load still leave the steady state
exactly where it was? Justify it from $f$ alone, without simulating.

**(iii)** Session 7 showed that negative autoregulation makes a gene reach its
steady state **faster**. A downstream load makes it **slower**. Do the two
cancel? You are not asked to compute it — say what you would compute, and what
result would settle it either way.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

**Q3b.** Two models, and they disagree.

**Protected**: the complex shields $X$ from removal. **Diluted**: the complex
is removed like everything else.

Set `X_PROTECTED` and `X_DILUTED` to the steady-state **free** $X$ in each.

Then set `SHIFT` to the fractional drop in the diluted model,
$(X_T - X_{ss})/X_T$, and `SHIFT_NAIVE` to what you would have predicted by
assuming the sites are completely full.

In [ ]:
K, GAMMA, KD, P_TOT = 5.0, 0.01, 50.0, 200.0
X_T = K / GAMMA

...

X_PROTECTED, X_DILUTED, SHIFT, SHIFT_NAIVE

In [ ]:
grader.check("q3b")

**Q3c.** Sweep the affinity. For
$K_d \in \{0.05,\ 0.5,\ 5,\ 50\}$ — three decades, all well below $X_T$ —
compute the **slowdown** in each model.

Set `SPEED_PROTECTED` and `SPEED_DILUTED` to lists of the four slowdown
factors, in that order of $K_d$.

Then set `FLAT` to the name of the model — the string `"protected"` or
`"diluted"` — in which the speed cost is nearly independent of $K_d$.

In [ ]:
KDS = [0.05, 0.5, 5.0, 50.0]

...

SPEED_PROTECTED, SPEED_DILUTED, FLAT

In [ ]:
grader.check("q3c")

<!-- BEGIN QUESTION -->

**Q3d.** Two short arguments.

**(i)** Write $\alpha(X) = 1/(1 + R(X))$ — the factor the load puts in front of
the entire right-hand side. At $X = 0$ this is $(1 + p_{\text{tot}}/K_d)^{-1}$,
which is the constant $\alpha$ Mishra *et al.* quote in Box 1.

A **constant** $\alpha$ rescales time uniformly, so bandwidth and rise time
would have to degrade by the same factor. Their bandwidth ratio is
$0.0140/0.0175 = 0.80$, which obliges a rise time $1/0.80 = 1.25\times$ longer.
The measured ratio is $698.9/396.1 = 1.76$.

Say what that gap tells you about treating $\alpha$ as a constant, and give
**one** reason to be careful before calling it a refutation. One short
paragraph.

**(ii)** Mishra *et al.* report rise times of **432.4 ± 8.9** minutes for the
buffered circuit with no load, against **396.1 ± 23.0** for the plain circuit,
standard deviations with $n = 3$ each. That is a 9% difference. **The paper does
not report it as a cost of insulation** — the two numbers sit in a list in the
Online Methods, and the paper's own buffered-against-unbuffered comparison runs
the other way. You are being asked whether that 9% is one.

Note also that the two rise times are measured to different targets: 90% of the
unbuffered mean for one and 90% of the buffered mean for the other.

Would you report a 9% insulation cost from these numbers? Say what you would
want before claiming it. One or two sentences, and you may compute something
if you wish.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

---
## Question 4 — Rejecting a disturbance without modeling it  *(14 points)*

The antithetic controller. A plant whose output is $X$, driven by a controller
species $Z_1$, with $Z_2$ produced in proportion to the output, and the two
controller species annihilating each other:

$$\frac{dX}{dt} = kZ_1 - \gamma X + d, \qquad
\frac{dZ_1}{dt} = \mu - \eta Z_1 Z_2 - \delta Z_1, \qquad
\frac{dZ_2}{dt} = \theta X - \eta Z_1 Z_2 - \delta Z_2$$

with $\mu = 10$, $\theta = 2$, $k = 1$, $\gamma = 0.5$, $\eta = 50$, and $d$ a
constant disturbance. $\delta$ is the dilution rate of the **controller
species** and is zero unless stated.

<!-- BEGIN QUESTION -->

**Q4a.** With $\delta = 0$, subtract the $Z_2$ equation from the
$Z_1$ equation.

**(i)** Show that the only steady state has $X = \mu/\theta$.

**(ii)** The annihilation rate $\eta$ is nowhere in your answer. Explain what
$\eta$ *does* control, given that it does not set the steady state.

**(iii)** This is why the motif is called an integrator. Say what quantity is
being integrated.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

**Q4b.** Write `simulate(delta, d, t_end=8000.0)` returning the
steady-state $X$ reached from $X = Z_1 = Z_2 = 0$. Use `solve_ivp` with
`method="LSODA"` and tight tolerances — the annihilation term is stiff.

Then set `X_NO_DIST` and `X_WITH_DIST` to the steady states at $\delta = 0$
for $d = 0$ and $d = 2$.

In [ ]:
MU, THETA, K_P, GAMMA_P, ETA = 10.0, 2.0, 1.0, 0.5, 50.0


def simulate(delta, d, t_end=8000.0):
    """Steady-state output of the antithetic loop, from rest."""
    ...


...

X_NO_DIST, X_WITH_DIST

In [ ]:
grader.check("q4b")

**Q4c.** Now switch on dilution of the controller species.

Set `ERRORS` to a list of the **set-point errors** $X_{ss} - \mu/\theta$ at
$\delta \in \{0,\ 0.001,\ 0.01,\ 0.05,\ 0.1\}$, all with $d = 1$.

> A note on names. "Adaptation error" is overloaded in this literature. In
> session 10 it meant final $\div$ peak for the feedforward loop — a
> dimensionless ratio read off the trajectory. Here and in Q5 it is a
> *difference*, $X_{ss} - \mu/\theta$, in units of $X$. Both make zero mean
> "perfect", so check which one a question is asking for.

Then: run the loop at $\delta = 0$ with $d = 5$ — a disturbance larger than
the plant needs to reach the target on its own. Set `SATURATED` to the output
at `t_end = 8000`.

**Now run it again at `t_end = 16000` and look at $Z_2$.** Set `SETTLES` to
`True` if the system reaches a steady state and `False` if it does not, and
set `Z2_RATIO` to $Z_2(16000) / Z_2(8000)$.

In [ ]:
DELTAS = [0.0, 0.001, 0.01, 0.05, 0.1]

...

ERRORS, SATURATED, SETTLES, Z2_RATIO

In [ ]:
grader.check("q4c")

---
## Question 5 — required for BioE 247, extra credit for BioE 147

**BioE 247** — part of the assignment, worth 8 of your 58
points.

**BioE 147** — optional, worth up to 8 points of extra credit on top of
50. You do not need it for full marks. It is not a harder version of the
same thing; it removes an assumption the earlier questions made, which is
where most of the interest is.

**Q5.** How does the set-point error $X_{ss} - \mu/\theta$ scale with the
dilution rate of the controller species?

Set `ERRS_SMALL` to the list of **absolute** errors at
$\delta \in \{0.0025,\ 0.005,\ 0.01,\ 0.02\}$, with $d = 1$, and `EXPONENT` to
the power $n$ in $|\text{error}| \propto \delta^{\,n}$ for small $\delta$ —
the slope of a straight line fitted to $\log|\text{error}|$ against
$\log\delta$.

Report the errors you actually fitted, not just the slope.

In [ ]:
SMALL = [0.0025, 0.005, 0.01, 0.02]

...

ERRS_SMALL, EXPONENT

In [ ]:
grader.check("q5")

<!-- BEGIN QUESTION -->

**Q5b.** Say *why* that exponent, from the steady-state algebra rather
than from the fit. Two or three sentences.

Start by subtracting the two controller equations at steady state, and say
what happens to the quantity that is left as $\delta \to 0$. Your answer
should also account for the **sign** you found in Q4c.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

---

### Before you submit

Restart the kernel and run every cell from the top. The autograder runs your
notebook in a clean environment, and a notebook that only works in the order
you happened to execute it will not pass.

**One closing thought, which is the set's spine.** You picked a medium in Q1
on two columns and could not fill the third. Q2 wrote a coupling down exactly
and found exactness was not enough. Q3 priced a coupling your diagram does not
draw at all. Q4 built the one thing that rejects a disturbance without
modeling it — and then watched dilution, which is the host's economy from
session 19, eat the guarantee.

The third column is not a number you look up. It is the rest of the course.